### NARX data interpretation

In [4]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

lag = 5
out_dim = 1
units = 32
n_epochs = 150
bs = 64
tol = 5
no_improve = 0

data = np.load(
    "/home/michel/Documents/machineLearningForControl/code/gym-unbalanced-disk-5SC28-group-24/disc-benchmark-files/training-val-test-data.npz"
)
y_raw = data["th"]
u_raw = data["u"]


def build_regressor_matrix(u, y, p):
    # stack lagged inputs and outputs into rows
    # each row: [u(k-p)...u(k-1), y(k-p)...y(k-1)] -> y(k)
    phi, targets = [], []
    for k in range(p, len(y)):
        phi.append(np.concatenate([u[k-p:k], y[k-p:k]]))
        targets.append(y[k])
    return np.array(phi), np.array(targets)

### Normalize data

In [ ]:
phi, targets = build_regressor_matrix(u_raw, y_raw, lag)

# hold out last 30% as test set (no shuffle, order matters for time series)
phi_train_val, phi_test, t_train_val, t_test = train_test_split(
    phi, targets, test_size=0.3, random_state=24, shuffle=False
)

# shuffle the remaining 70% and split into train/val
phi_train, phi_val, t_train, t_val = train_test_split(
    phi_train_val, t_train_val, test_size=0.35, random_state=24, shuffle=True
)

# normalise using train stats only
x_mu = phi_train.mean(axis=0)
x_sig = phi_train.std(axis=0)
y_mu = t_train.mean()
y_sig = t_train.std()

phi_train = (phi_train - x_mu) / x_sig
phi_val   = (phi_val   - x_mu) / x_sig
phi_test  = (phi_test  - x_mu) / x_sig

t_train = (t_train - y_mu) / y_sig
t_val   = (t_val   - y_mu) / y_sig
t_test  = (t_test  - y_mu) / y_sig

# to tensors
to_t = lambda a: torch.tensor(a, dtype=torch.float32)
phi_train, phi_val, phi_test = to_t(phi_train), to_t(phi_val), to_t(phi_test)
t_train, t_val, t_test       = to_t(t_train),   to_t(t_val),   to_t(t_test)

input_size = phi_train.shape[1]